# Qué es “estacionalidad”

Un patrón que se repite cada año (meses, semanas, estaciones).
Ejemplo típico: en Tenerife (y casi cualquier sitio) temperatura y muchas veces muertes tienen forma anual; PM también puede tenerla.

## Por qué el “absoluto” engaña

Porque estás mezclando dos cosas en la misma señal:

Efecto estacional (mes del año)

Efecto real que te interesa (calima/PM/alertas)

Si en invierno suben muertes y también sube PM10, una correlación “a pelo” puede salir alta aunque PM10 no sea la causa principal. Es el clásico confounding: la estación está empujando ambas variables a la vez.

Tu ejemplo real de hoy:

muertes medias por mes cambian ~40

pm10 medio por mes también es más alto en 12–1–2
→ el absoluto puede estar midiendo “invierno vs verano”.

## Cómo identificar estacionalidad (rápido)

Por mes:
mean(y | month) y miras max-min. Si es grande, hay estacionalidad.

Por semana del año: mejor resolución (1–52/53), mismo concepto.

Visual: serie temporal con 1–2 años te lo grita.

## Cómo controlarla (3 niveles)
### Nivel 1: Comparar “dentro de la estación”

No compares “yellow vs non-yellow” global.
Compara dentro de mes (o dentro de week-of-year).
Conceptualmente: “¿En enero, cuando hay más PM, también suben muertes respecto a otros eneros?”

In [2]:
# Calcula mean(y|month) y max-min para deaths, tmax, pm10.
import pandas as pd
from pathlib import Path

# 1) Carga dataset Gran Canaria (ajusta si tu parquet está en otra ruta)
# data\processed\gran_canaria\master\master_gcan_2015_2024.parquet
fp = Path(r"C:\dev\projects\climate_mortality\data\processed\gran_canaria\master\master_gcan_2015_2024.parquet")
df = pd.read_parquet(fp)



In [3]:
# 2) Nos quedamos con lo mínimo
cols = ["week_start", "deaths_week", "tmax_c_mean", "PM10"]
x = df[cols].copy()

# 3) Función: mean(y|month) y rango (max-min) de esas medias mensuales
def month_seasonality(series, dates):
    m = series.groupby(dates.dt.month).mean()
    return float(m.min()), float(m.max()), float(m.max()-m.min())

for v in ["deaths_week", "tmax_c_mean", "PM10"]:
    mn, mx, rng = month_seasonality(x[v], x["week_start"])
    print(v, "max-min =", round(rng, 2), "| min=", round(mn,2), "max=", round(mx,2))


deaths_week max-min = 40.37 | min= 121.29 max= 161.66
tmax_c_mean max-min = 7.02 | min= 21.64 max= 28.66
PM10 max-min = 75.9 | min= 36.41 max= 112.31


### si haces correlación muertes vs PM10 en bruto, te va a salir “fuerte” aunque parte sea simplemente mes del año. Porque PM10 cambia muchísimo con la estación.

In [5]:
# ver qué meses son los picos y valles de PM10 y muertes (para confirmar si coinciden)
m_deaths = x.groupby(x["week_start"].dt.month)["deaths_week"].mean()
m_pm10   = x.groupby(x["week_start"].dt.month)["PM10"].mean()

print("Deaths highest months:\n", m_deaths.sort_values(ascending=False).head(3))
print("Deaths lowest months:\n",  m_deaths.sort_values().head(3))

print("\nPM10 highest months:\n",  m_pm10.sort_values(ascending=False).head(3))
print("PM10 lowest months:\n",    m_pm10.sort_values().head(3))
print("="*50)
print("correlation weeks (absolute):", x["deaths_week"].corr(x["PM10"]))
print("correlation months (absolute):", m_deaths.corr(m_pm10))

Deaths highest months:
 week_start
1     161.658537
2     149.891892
12    145.050000
Name: deaths_week, dtype: float64
Deaths lowest months:
 week_start
7    121.292683
6    122.783784
9    126.078947
Name: deaths_week, dtype: float64

PM10 highest months:
 week_start
2     112.305019
1      81.435540
12     80.654167
Name: PM10, dtype: float64
PM10 lowest months:
 week_start
6    36.405405
5    39.142857
7    40.313589
Name: PM10, dtype: float64
correlation weeks (absolute): 0.2521860763192113
correlation months (absolute): 0.8373672876933687


### Nivel 2: Anomalías (quita la media estacional)

La idea: a cada semana le quitas “lo normal para ese mes”.
y_anom = y - mean(y | month)
x_anom = x - mean(x | month)

Luego miras relación entre y_anom y x_anom.
Eso responde: “cuando PM está por encima de lo típico de ese mes, ¿muertes también están por encima?”

In [6]:
deaths_anom = x["deaths_week"] - x.groupby(x["week_start"].dt.month)["deaths_week"].transform("mean")
pm10_anom   = x["PM10"]        - x.groupby(x["week_start"].dt.month)["PM10"].transform("mean")

deaths_anom.corr(pm10_anom)

np.float64(0.1071615931943427)

### anomalías por week-of-year (woy)

Esto suele bajar aún más el confounding que “mes”.

In [7]:
woy = x["week_start"].dt.isocalendar().week.astype(int)
deaths_anom_w = x["deaths_week"] - x.groupby(woy)["deaths_week"].transform("mean")
pm10_anom_w   = x["PM10"]        - x.groupby(woy)["PM10"].transform("mean")

deaths_anom_w.corr(pm10_anom_w)

np.float64(0.09783346411289442)

corr ~ 0.098 confirma: al controlar estacionalidad fina, queda una relación pequeña entre PM10 y muertes (mucho menor que la señal mensual absoluta 0.84).

En absoluto parece haber relación; al desestacionalizar por woy cae a ~0.10 → confounding fuerte.

### Seasonality and confounding (Gran Canaria)

In the raw weekly data, deaths and PM10 show a moderate positive correlation (≈0.25). However, monthly averages of deaths and PM10 are very strongly correlated (≈0.84), indicating that both variables share a strong seasonal pattern.

After de-seasonalising by month and by week-of-year (anomalies), the remaining within-season correlation drops to a small value (≈0.10). This suggests that much of the apparent association in absolute values is driven by shared seasonality rather than week-to-week co-movement within the same time of year.

These results are descriptive and do not support causal claims about PM10; further modelling (lags, non-linear thresholds, and additional confounders) would be required to assess whether any residual effect remains.

### 3 claims NO válidos (no los puedes decir)

❌ “PM10 causa más muertes.”
Por qué: incluso con desestacionalización solo tienes asociación (y pequeña). Además faltan confusores (virus, otros contaminantes, meteorología). No hay diseño causal.

❌ “Como la correlación anómala es baja (~0.10), PM10 no afecta a la mortalidad.”
Por qué: “baja” no es “cero” y podría haber efectos no lineales (umbral), lags, o medición ruidosa. También puede haber falta de potencia o mala especificación.

❌ “El efecto real de PM10 es exactamente 0.10.”
Por qué: 0.10 es una estimación puntual de correlación en tu muestra; no es un parámetro fijo del mundo. Tiene incertidumbre y depende de periodo, definición, agregación semanal, etc.

### Efecto estacional compartido: fuerte.
**Relación residual** (PM10 por encima de lo típico del mes ↔ muertes por encima de lo típico del mes): pequeña (y no implica causalidad).

In [6]:
# CAP

# 1) Anomalía mensual de muertes
x = df[["week_start", "deaths_week"]].copy()
x["deaths_anom"] = x["deaths_week"] - x.groupby(x["week_start"].dt.month)["deaths_week"].transform("mean")

# 2) Columna CAP yellow (ajusta el nombre aquí)
candidates = [c for c in df.columns if "yellow" in c.lower() or "dust" in c.lower() or "cap" in c.lower()]
# print(candidates)
cap_col = "cap_dust_yellow_plus_week"
# 3) Comparación yellow vs non-yellow
x = df[["week_start", "deaths_week", "cap_dust_yellow_plus_week"]].copy()

# anomalía mensual de muertes
x["deaths_anom"] = x["deaths_week"] - x.groupby(x["week_start"].dt.month)["deaths_week"].transform("mean")

# compara anomalías por CAP dust yellow+
g = x.groupby("cap_dust_yellow_plus_week")["deaths_anom"]

out = pd.DataFrame({
    "n": g.size(),
    "mean_deaths_anom": g.mean(),
    "median_deaths_anom": g.median(),
    "p25": g.quantile(0.25),
    "p75": g.quantile(0.75),
}).rename_axis("cap_dust_yellow_plus_week")

out

,n,mean_deaths_anom,median_deaths_anom,p25,p75
cap_dust_yellow_plus_week,,,,,
0.0,317,3.399559,3.108108,-7.783784,13.631579
1.0,25,10.755003,10.108108,1.108108,20.707317


#### “Tras eliminar estacionalidad mensual, las semanas con CAP dust yellow+ muestran ~+11 muertes (media) y ~+10 (mediana) por encima de lo esperado.”

### Nivel 3: Ajuste explícito en modelo

Objetivo: estimar el efecto de PM10 o CAP controlando por estación.


In [7]:
import statsmodels.formula.api as smf

x = df[["week_start","deaths_week","PM10","cap_dust_yellow_plus_week"]].copy()
x["month"] = x["week_start"].dt.month
x["year"]  = x["week_start"].dt.year

# 1) Efecto PM10 controlando por mes + año
m1 = smf.ols("deaths_week ~ PM10 + C(month) + C(year)", data=x).fit()
print(m1.summary().tables[1])

# 2) Efecto CAP dust yellow+ controlando por mes + año
m2 = smf.ols("deaths_week ~ cap_dust_yellow_plus_week + C(month) + C(year)", data=x).fit()
print(m2.summary().tables[1])

                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept         148.5145     14.386     10.324      0.000     120.242     176.787
C(month)[T.2]     -11.1099      3.202     -3.470      0.001     -17.402      -4.818
C(month)[T.3]     -21.9412      3.177     -6.906      0.000     -28.185     -15.698
C(month)[T.4]     -29.9289      3.168     -9.448      0.000     -36.154     -23.704
C(month)[T.5]     -34.0640      3.136    -10.864      0.000     -40.226     -27.902
C(month)[T.6]     -38.3358      3.223    -11.893      0.000     -44.671     -32.001
C(month)[T.7]     -39.6185      3.132    -12.650      0.000     -45.773     -33.464
C(month)[T.8]     -32.0040      3.150    -10.160      0.000     -38.194     -25.814
C(month)[T.9]     -34.9538      3.190    -10.957      0.000     -41.223     -28.684
C(month)[T.10]    -34.8337      3.114    -11.185      0.000     -40.954     

### Modelo 1: deaths ~ PM10 + mes + año

PM10 coef = 0.0034, p = 0.768 → cero evidencia de efecto lineal contemporáneo una vez controlas mes/año.

Los dummies de mes son enormes y muy significativos → la estacionalidad domina (lo que ya viste con max–min).

Años: casi ninguno significativo salvo 2022 (p=0.041) → posible “shift” ese año, pero no es el foco ahora.

Lectura: la correlación bruta muertes–PM10 en GC era sobre todo estación compartida.

### Modelo 2: deaths ~ CAP dust yellow+ + mes + año

cap_dust_yellow_plus_week coef = +5.37, p = 0.086
→ señal positiva (más muertes en semanas yellow+), pero no pasa 0.05; es “marginal”.

Meses siguen mandando.

Por qué te puede salir +5 aquí pero +10 en anomalías:
Tu método de anomalías solo controla mes. Este modelo además controla año, y eso puede comerse parte del efecto (y también cambia el ajuste por la forma en que se ponderan semanas y se maneja variación).

In [8]:
x = df[["week_start","deaths_week","PM10","cap_dust_yellow_plus_week"]].copy()
x["woy"]  = x["week_start"].dt.isocalendar().week.astype(int)
x["year"] = x["week_start"].dt.year

m_cap_woy = smf.ols("deaths_week ~ cap_dust_yellow_plus_week + C(woy) + C(year)", data=x).fit()
m_pm_woy  = smf.ols("deaths_week ~ PM10 + C(woy) + C(year)", data=x).fit()

print(m_cap_woy.summary().tables[1])
print(m_pm_woy.summary().tables[1])

                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                   139.0948      4.719     29.478      0.000     129.807     148.383
C(woy)[T.2]                   4.7929      7.906      0.606      0.545     -10.770      20.356
C(woy)[T.3]                   3.7929      7.906      0.480      0.632     -11.770      19.356
C(woy)[T.4]                   1.3129      7.931      0.166      0.869     -14.298      16.923
C(woy)[T.5]                  -1.2272      7.919     -0.155      0.877     -16.815      14.361
C(woy)[T.6]                  -0.0204      7.931     -0.003      0.998     -15.631      15.590
C(woy)[T.7]                 -15.0405      7.906     -1.902      0.058     -30.603       0.522
C(woy)[T.8]                 -13.3738      7.906     -1.692      0.092     -28.937       2.189
C(woy)[T.9]                 -10.7071      7.906     -1.354  

In [9]:
x = df[["week_start","deaths_week","PM10","cap_dust_yellow_plus_week"]].copy()
x = x.sort_values("week_start")
x["woy"]  = x["week_start"].dt.isocalendar().week.astype(int)
x["year"] = x["week_start"].dt.year

x["PM10_lag1"] = x["PM10"].shift(1)
x["CAP_lag1"]  = x["cap_dust_yellow_plus_week"].shift(1)

m_pm_lag1  = smf.ols("deaths_week ~ PM10_lag1 + C(woy) + C(year)", data=x).fit()
m_cap_lag1 = smf.ols("deaths_week ~ CAP_lag1 + C(woy) + C(year)", data=x).fit()

print("PM10_lag1 coef:", m_pm_lag1.params["PM10_lag1"], "p:", m_pm_lag1.pvalues["PM10_lag1"])
print("CAP_lag1  coef:", m_cap_lag1.params["CAP_lag1"],  "p:", m_cap_lag1.pvalues["CAP_lag1"])

PM10_lag1 coef: 0.0025118110082142948 p: 0.8321662483605604
CAP_lag1  coef: 10.201471896143508 p: 0.0016053079041288237


Hay asociación temporal consistente con un retardo: episodio/alerta → incremento en muertes la semana siguiente.

PM10 como variable continua no está capturando lo mismo (o está muy ruidosa / mal alineada temporalmente / o el efecto no es lineal).

In [10]:
x = df[["week_start","deaths_week","PM10","cap_dust_yellow_plus_week"]].copy()
x = x.sort_values("week_start")
x["woy"]  = x["week_start"].dt.isocalendar().week.astype(int)
x["year"] = x["week_start"].dt.year

for L in [0,1,2]:
    x[f"CAP_lag{L}"] = x["cap_dust_yellow_plus_week"].shift(L)
    m = smf.ols(f"deaths_week ~ CAP_lag{L} + C(woy) + C(year)", data=x).fit()
    print(L, "coef", round(m.params[f"CAP_lag{L}"],2), "p", m.pvalues[f"CAP_lag{L}"])

0 coef 3.88 p 0.23292978419729565
1 coef 10.2 p 0.0016053079041288237
2 coef 17.0 p 8.956823580643022e-08


La señal de CAP dust yellow+ en GC no es contemporánea (no se ve en la misma semana).

Aparece con retardo 1–2 semanas, y en tus datos es más fuerte a 2 semanas.

Esto es compatible con:

retrasos plausibles en efectos respiratorios/cardiovasculares (no siempre “instantáneo”),

o con cómo estás definiendo semanas: la exposición (alerta) puede ocurrir al final de una semana y las muertes “caen” en la(s) siguiente(s),

o con que los episodios duren varios días/semanas y el “lag” está capturando persistencia

In [11]:
x = df[["week_start","deaths_week","cap_dust_yellow_plus_week"]].copy()
x = x.sort_values("week_start")
x["woy"]  = x["week_start"].dt.isocalendar().week.astype(int)
x["year"] = x["week_start"].dt.year

x["CAP_lag1"] = x["cap_dust_yellow_plus_week"].shift(1)
x["CAP_lag2"] = x["cap_dust_yellow_plus_week"].shift(2)

m12 = smf.ols("deaths_week ~ CAP_lag1 + CAP_lag2 + C(woy) + C(year)", data=x).fit()
print("lag1:", m12.params["CAP_lag1"], "p:", m12.pvalues["CAP_lag1"])
print("lag2:", m12.params["CAP_lag2"], "p:", m12.pvalues["CAP_lag2"])

lag1: 7.822135295993823 p: 0.012409202865713035
lag2: 15.832691508015182 p: 6.151480126829064e-07


## Controlando por semana del año y año:

La alerta CAP dust yellow+ se asocia con exceso de muertes principalmente a 2 semanas vista, y en menor medida a 1 semana.

El efecto contemporáneo (lag0) no aparece.
    
“En Gran Canaria, el indicador CAP de polvo (yellow+) muestra una asociación retardada con mortalidad semanal: +~16 muertes a 2 semanas (p<1e-6) y +~8 a 1 semana (p≈0.01), tras ajustar estacionalidad fina (woy) y año.”

## Checks

In [12]:
import statsmodels.formula.api as smf

x = df[["week_start","deaths_week","cap_dust_yellow_plus_week","tmax_c_mean"]].copy()
x = x.sort_values("week_start")
x["woy"]  = x["week_start"].dt.isocalendar().week.astype(int)
x["year"] = x["week_start"].dt.year

x["CAP_lag1"] = x["cap_dust_yellow_plus_week"].shift(1)
x["CAP_lag2"] = x["cap_dust_yellow_plus_week"].shift(2)

m = smf.ols("deaths_week ~ CAP_lag1 + CAP_lag2 + tmax_c_mean + C(woy) + C(year)", data=x).fit()

print("lag1:", round(m.params["CAP_lag1"],2), "p:", m.pvalues["CAP_lag1"])
print("lag2:", round(m.params["CAP_lag2"],2), "p:", m.pvalues["CAP_lag2"])
print("tmax:", round(m.params["tmax_c_mean"],3), "p:", m.pvalues["tmax_c_mean"])

lag1: 7.39 p: 0.01889705370178628
lag2: 15.61 p: 1.0088345388760425e-06
tmax: 0.657 p: 0.33633549919638983


El efecto retardado de CAP dust yellow+ sobre muertes no parece ser solo temperatura/estación, porque:

ya controlabas woy y año

y al meter tmax explícitamente, los coeficientes CAP casi no se mueven y siguen significativos.

In [13]:
m.conf_int().loc[["CAP_lag1","CAP_lag2","tmax_c_mean"]]

,0,1
CAP_lag1,1.228885,13.544310
CAP_lag2,9.467604,21.755584
tmax_c_mean,-0.686199,2.000782


En Gran Canaria, tras ajustar por estacionalidad fina (semana del año) y año, las semanas con CAP dust yellow+ se asocian con un exceso de mortalidad 1–2 semanas después; el efecto a 2 semanas es especialmente robusto (IC95% > 0).

## Porqué p90/95 es útil
- Outliers sin drama
- Si PM10 tiene semanas de 300 por un episodio raro o un fallo, el max se dispara. p95 te describe mejor el “alto habitual”.

#### Definir episodios / thresholds
“Semana de PM10 extremo = PM10 > p95” es una regla clara y reproducible (mejor que “alto a ojo”).

#### Comparar distribuciones entre islas / periodos
p95 de PM10 en GC vs TFE te dice cuál tiene “colas” más pesadas aunque las medias sean parecidas.

#### Robustez en reporting
En un informe, decir “p95=112” comunica riesgo/colas mejor que solo media=60.

#### p90 / p95 = umbral alto pero representativo:

p90: “el 90% de las semanas está por debajo de esto”

p95: “el 95% está por debajo”

In [14]:
pm = df["PM10"].dropna()

stats = {
    "mean": pm.mean(),
    "median": pm.median(),
    "p90": pm.quantile(0.90),
    "p95": pm.quantile(0.95),
    "p99": pm.quantile(0.99),
    "max": pm.max(),
}

stats

{'mean': np.float64(55.545394803356594),
 'median': np.float64(41.142857142857146),
 'p90': np.float64(84.42857142857143),
 'p95': np.float64(125.5),
 'p99': np.float64(308.52857142857215),
 'max': np.float64(934.4285714285714)}

In [15]:
(stats["p95"] / stats["median"], stats["max"] / stats["p95"])

(np.float64(3.050347222222222), np.float64(7.445645987478657))

en GC, **PM10 no es homogéneo**, tiene **pocas semanas que dominan la distribución.**

In [16]:
x = df[["week_start","PM10"]].copy()
x = x.sort_values("week_start")

thr = x["PM10"].quantile(0.95)
x["is_ep"] = x["PM10"] > thr

# contar semanas episodio
n_weeks = int(x["is_ep"].sum())
print("p95 threshold:", thr)
print("weeks above p95:", n_weeks)

# identificar episodios como rachas consecutivas
# episodio_id incrementa cuando empieza una racha (0->1)
start = x["is_ep"] & (~x["is_ep"].shift(1, fill_value=False))
x["episode_id"] = start.cumsum()
x.loc[~x["is_ep"], "episode_id"] = pd.NA

episodes = (
    x.dropna(subset=["episode_id"])
     .groupby("episode_id")
     .agg(
         start_week=("week_start","min"),
         end_week=("week_start","max"),
         weeks=("week_start","size"),
         pm10_peak=("PM10","max"),
         pm10_mean=("PM10","mean"),
     )
     .reset_index(drop=True)
)

print("\n#episodes:", len(episodes))
print("duration (weeks) mean:", episodes["weeks"].mean())
print("duration (weeks) max:", episodes["weeks"].max())

episodes.sort_values(["weeks","pm10_peak"], ascending=False).head(10)

p95 threshold: 125.5
weeks above p95: 24

#episodes: 15
duration (weeks) mean: 1.6
duration (weeks) max: 5


,start_week,end_week,weeks,pm10_peak,pm10_mean
7,2022-01-10,2022-02-07,5,509.428571,262.400000
13,2024-01-22,2024-02-05,3,202.285714,174.523810
4,2020-02-17,2020-02-24,2,934.428571,573.642857
10,2023-02-06,2023-02-13,2,361.285714,245.785714
0,2016-12-19,2016-12-26,2,176.285714,151.142857
12,2023-12-11,2023-12-11,1,361.571429,361.571429
6,2021-02-15,2021-02-15,1,354.428571,354.428571
11,2023-10-02,2023-10-02,1,247.714286,247.714286
14,2024-12-16,2024-12-16,1,237.000000,237.000000
3,2020-02-03,2020-02-03,1,235.285714,235.285714


In [17]:
# Dos métricas útiles más (para describir “tamaños pequeños” bien):
# cuántos episodios son de 1 sola semana (ruido/pico)
# cuántos son de ≥2 semanas (episodio robusto)
episodes["weeks"].value_counts().sort_index()

weeks
1    10
2     3
3     1
5     1
Name: count, dtype: int64

In [18]:
# Coincidencia de episodios con CAP semanal
x = df[["week_start","PM10","cap_dust_yellow_plus_week"]].copy().sort_values("week_start")

# 1) quedarse solo con periodo con CAP disponible (ajusta fecha exacta si tu cobertura empieza otra semana)
x = x[x["week_start"] >= "2018-01-01"]

# 2) quitar semanas sin dato CAP (muy importante si hay NaN)
x = x.dropna(subset=["cap_dust_yellow_plus_week", "PM10"])

thr = x["PM10"].quantile(0.95)
x["pm95"] = (x["PM10"] > thr).astype(int)
x["cap"]  = (x["cap_dust_yellow_plus_week"] == 1).astype(int)

pd.crosstab(x["pm95"], x["cap"], rownames=["PM10>p95"], colnames=["CAP yellow+"])

CAP yellow+,0,1
PM10>p95,,
0,310,14
1,7,11


In [19]:
# Rachas (episodios PM10) y ver cuantos coinciden


x = df[["week_start","PM10","cap_dust_yellow_plus_week"]].copy().sort_values("week_start")
x = x[x["week_start"] >= "2018-01-01"]
x = x.dropna(subset=["cap_dust_yellow_plus_week", "PM10"])

thr = x["PM10"].quantile(0.95)
x["pm95"] = (x["PM10"] > thr).astype(int)
x["cap"]  = (x["cap_dust_yellow_plus_week"] == 1).astype(int)
x["both"] = ((x["pm95"] == 1) & (x["cap"] == 1)).astype(int)

# episodios PM95 (rachas)
start = (x["pm95"] == 1) & (x["pm95"].shift(1, fill_value=0) == 0)
x["pm95_episode_id"] = start.cumsum()
x.loc[x["pm95"] == 0, "pm95_episode_id"] = pd.NA

pm_eps = (
    x.dropna(subset=["pm95_episode_id"])
     .groupby("pm95_episode_id")
     .agg(weeks=("week_start","size"))
)

x["pm95_ep_weeks"] = x["pm95_episode_id"].map(pm_eps["weeks"])
x["pm95_ep_type"]  = pd.cut(x["pm95_ep_weeks"], bins=[0,1,999], labels=["1-week","2+ weeks"])

# ENTRE LAS SEMANAS QUE COINCIDEN
x.loc[x["both"]==1, "pm95_ep_type"].value_counts()

pm95_ep_type
2+ weeks    6
1-week      5
Name: count, dtype: int64

## Conclusion
### Episodio = racha (no semana suelta).
CAP define el “evento” y PM10 cuantifica la “severidad”.
Si usamos PM10 puro,  se exige ≥2 semanas o al menos reporta sensibilidad (1-week vs 2+).

In [20]:
###### ESQUEMA DE COLUMNAS ESTANDAR PARA GUARDAR EPISODIOS EN UN DATAFRAME -CSV / VALIDO PARA TODAS LAS ISLAS
import pandas as pd
from pathlib import Path

# ---------- CONFIG ----------
OUTDIR = Path("reports/tables/island/GC")
OUTDIR.mkdir(parents=True, exist_ok=True)

x = df[["week_start","deaths_week","PM10","cap_dust_yellow_plus_week"]].copy()
x = x.sort_values("week_start")

# CAP válido desde 2018 (ajusta si tu cobertura empieza en otra fecha)
x = x[x["week_start"] >= "2018-01-01"].dropna(subset=["PM10","cap_dust_yellow_plus_week","deaths_week"])

# ---------- 1) EPISODIOS CAP (rachas CAP==1) ----------
x["cap"] = (x["cap_dust_yellow_plus_week"] == 1).astype(int)

cap_start = (x["cap"] == 1) & (x["cap"].shift(1, fill_value=0) == 0)
x["cap_episode_id"] = cap_start.cumsum()
x.loc[x["cap"] == 0, "cap_episode_id"] = pd.NA

cap_eps = (
    x.dropna(subset=["cap_episode_id"])
     .groupby("cap_episode_id")
     .agg(
         start_week=("week_start","min"),
         end_week=("week_start","max"),
         duration_weeks=("week_start","size"),
         pm10_peak=("PM10","max"),
         pm10_mean=("PM10","mean"),
         deaths_mean=("deaths_week","mean"),
         deaths_sum=("deaths_week","sum"),
     )
     .reset_index(drop=True)
)

cap_fp = OUTDIR / "episodes_cap_dust_gcan_2018_2024.csv"
cap_eps.to_csv(cap_fp, index=False)

# ---------- 2) EPISODIOS PM10 p95 (rachas PM10>p95) ----------
thr = x["PM10"].quantile(0.95)
x["pm95"] = (x["PM10"] > thr).astype(int)

pm_start = (x["pm95"] == 1) & (x["pm95"].shift(1, fill_value=0) == 0)
x["pm95_episode_id"] = pm_start.cumsum()
x.loc[x["pm95"] == 0, "pm95_episode_id"] = pd.NA

pm_eps = (
    x.dropna(subset=["pm95_episode_id"])
     .groupby("pm95_episode_id")
     .agg(
         start_week=("week_start","min"),
         end_week=("week_start","max"),
         duration_weeks=("week_start","size"),
         pm10_peak=("PM10","max"),
         pm10_mean=("PM10","mean"),
         any_cap=("cap","max"),  # si el episodio PM95 coincide con CAP en alguna semana
     )
     .reset_index(drop=True)
)

pm_eps["episode_type"] = pm_eps["duration_weeks"].apply(lambda n: "2+ weeks" if n >= 2 else "1-week")
pm_eps["pm95_threshold"] = float(thr)

pm_fp = OUTDIR / "episodes_pm10_p95_gcan_2018_2024.csv"
pm_eps.to_csv(pm_fp, index=False)

print("Wrote:")
print(" -", cap_fp)
print(" -", pm_fp)
print("PM10 p95 threshold used:", float(thr))

Wrote:
 - reports\tables\island\GC\episodes_cap_dust_gcan_2018_2024.csv
 - reports\tables\island\GC\episodes_pm10_p95_gcan_2018_2024.csv
PM10 p95 threshold used: 154.44999999999976


## Interpretar lags 0-1
Porque correlacion no implica causalida; que evidencia extra haria falta.

## Qué significa “lag”

Un **lag** es “mover” la exposición hacia atrás para ver si afecta **después**.

Si tu outcome es muertes en la semana **t**:

* **lag0** usa la exposición de la **misma semana t**
* **lag1** usa la exposición de la **semana anterior (t−1)**
* **lag2** usa la exposición de **hace dos semanas (t−2)**

## Interpretación en tu modelo (CAP)

Cuando estimas:

`deaths_week(t) ~ CAP_lag1 + ...`

estás preguntando:

> “¿En semanas donde **la semana pasada** hubo CAP yellow+, hay **más muertes esta semana**, comparado con semanas donde la semana pasada no lo hubo, controlando por estacionalidad y año?”

El coeficiente se lee así:

* Si `CAP_lag1 = +10`, significa:

  * **~10 muertes más** en la semana **t** cuando en **t−1** hubo CAP yellow+, *manteniendo constantes* woy/año (y lo que metas).

## Por qué lag0 puede ser nulo pero lag1/lag2 sí

Porque el efecto puede no ser inmediato:

* el episodio ocurre a mitad/final de la semana → las muertes “caen” en la siguiente
* el impacto en salud tarda días → se ve en 1–2 semanas

## Importante: qué NO significa

* No prueba causalidad por sí solo.
* Y si los episodios duran varias semanas, **lag1 y lag2 están correlacionados**: por eso metimos lag1+lag2 juntos para ver cuál aporta más.

En caso GC:

* lag0 no significativo
* lag1 significativo
* lag2 muy significativo
  → sugiere que el “timing” del efecto está más **desplazado** hacia 1–2 semanas después del episodio.


## Por qué no implica causalidad

### 1) Confusores no medidos

Puede existir una tercera variable que:

* aumenta la probabilidad de CAP yellow+
* y también aumenta muertes 1–2 semanas después

Ejemplos plausibles:

* **gripe/virus estacionales**, olas de infecciones
* **otros contaminantes** (NO₂, O₃) o mezcla de contaminación urbana
* **meteorología** más completa (viento, estabilidad, lluvia, presión), no solo tmax
* **eventos sociales** (p.ej. periodos vacacionales cambian movilidad/salud)

Aunque controles woy/año, puede quedar confusión “dentro de la semana del año” (p.ej. un brote concreto de un año).

### 2) Medición imperfecta / proxy

CAP es un **indicador** (alerta), no exposición individual real. Puede estar correlacionado con lo real, pero también con cosas administrativas/meteorológicas.

### 3) Duración de episodios y autocorrelación

Los episodios duran varias semanas → lag1 y lag2 están correlacionados → puedes capturar “periodos malos” más que un mecanismo causal puntual.

### 4) Reverse causality no aplica mucho aquí, pero sí “timing ambiguity”

Muertes no causan CAP, pero sí puede haber:

* desalineación entre cuándo ocurre el polvo y cómo se agrega semanalmente,
* o efectos que caen en bordes de semana y se “mueven” a lag1/lag2 por el calendario.

### 5) Multiplicidad / búsqueda

Si pruebas muchos lags, muchas variables, muchas islas, **alguna** saldrá “significativa” por azar si no controlas el proceso (pre-registro o corrección).

---

## Qué evidencia haría falta (de menos a más fuerte)

### A) Robustez interna (mínimo exigible)

* Replicar el efecto **en otras islas** y/o **en otros periodos**
* Mostrar que el efecto aguanta:

  * otros controles (viento/lluvia/estabilidad, otros contaminantes)
  * otras definiciones de episodio (p90/p95, CAP, etc.)
  * otras especificaciones (lags 0–3, distributed lag)
* Pruebas placebo:

  * **lead test**: usar CAP *futuro* (t+1) no debería “predecir” muertes en t
  * outcomes placebo si tienes alguno

### B) Evidencia de “mecanismo” (muy convincente)

* Ver que el efecto es mayor en subgrupos plausibles:

  * mayores, respiratorio/cardiovascular, etc. (si tienes causas o grupos de edad)
* Relación dosis–respuesta:

  * episodios más intensos → más exceso (pico PM10 mayor → mayor efecto)
* Coherencia temporal:

  * pico del efecto en 1–2 semanas consistente en varias islas/periodos

### C) Diseño cuasi-experimental (lo más cercano a causalidad sin RCT)

* **Interrupted time series / event study** alrededor de episodios CAP:

  * comparar semanas cercanas al episodio contra semanas similares sin episodio
* **Difference-in-differences**:

  * comparar islas afectadas vs menos afectadas en el mismo periodo
* **Instrumental variables** (difícil pero ideal):

  * usar variables meteorológicas que “empujan” polvo (dirección del viento, índices de intrusión) como instrumento, si cumplen supuestos.

### D) Datos más finos (sube mucho la credibilidad)

* Pasar de semanal a **diario** (si puedes) para:

  * alinear mejor exposición y muertes
  * ver la forma del lag (0–14 días) con más resolución

---

### En GC, lo que más sube credibilidad rápido

1. **Lead test** (CAP(t+1) no debe predecir deaths(t))
2. Añadir **meteorología clave** (viento/lluvia/estabilidad) si la tienes
3. Replicar el patrón lag1–lag2 en **Tenerife** u otra isla



In [21]:
import statsmodels.formula.api as smf

x = df[["week_start","deaths_week","cap_dust_yellow_plus_week"]].copy().sort_values("week_start")
x = x[x["week_start"] >= "2018-01-01"].dropna()

x["woy"]  = x["week_start"].dt.isocalendar().week.astype(int)
x["year"] = x["week_start"].dt.year

# "leads" = CAP del futuro
x["CAP_lead1"] = x["cap_dust_yellow_plus_week"].shift(-1)
x["CAP_lead2"] = x["cap_dust_yellow_plus_week"].shift(-2)

m_lead1 = smf.ols("deaths_week ~ CAP_lead1 + C(woy) + C(year)", data=x).fit()
m_lead2 = smf.ols("deaths_week ~ CAP_lead2 + C(woy) + C(year)", data=x).fit()

print("lead1 coef:", round(m_lead1.params["CAP_lead1"],2), "p:", m_lead1.pvalues["CAP_lead1"])
print("lead2 coef:", round(m_lead2.params["CAP_lead2"],2), "p:", m_lead2.pvalues["CAP_lead2"])

lead1 coef: 0.58 p: 0.8591404822726666
lead2 coef: 0.27 p: 0.9346777759943753


#### Probamos con dataset calima island

In [22]:
fp = Path(r"C:\dev\projects\heat_mortality_analysis\data\processed\gran_canaria\calima\calima_proxy_weekly_gcan_2015_2024_v2.parquet")
df = pd.read_parquet(fp)
print(df.columns.tolist())

['week_start', 'deaths_week', 'PM10', 'PM2.5', 'humidity_mean', 'pressure_hpa_mean', 'low_vis_any_week', 'cap_dust_yellow_plus_week', 'cap_dust_level_max_week', 'calima_dai_flag', 'pm10_p90_flag', 'pm10_p95_flag', 'hum_low_flag', 'low_vis_flag', 'pressure_high_flag', 'calima_proxy_score_v2', 'calima_proxy_level_v2']


In [23]:
x = df[["week_start","deaths_week","calima_proxy_score_v2"]].copy().sort_values("week_start")
x = x[x["week_start"] >= "2018-01-01"].dropna()

x["woy"]  = x["week_start"].dt.isocalendar().week.astype(int)
x["year"] = x["week_start"].dt.year

# lags 0-1-2
x["cal_lag0"] = x["calima_proxy_score_v2"]
x["cal_lag1"] = x["calima_proxy_score_v2"].shift(1)
x["cal_lag2"] = x["calima_proxy_score_v2"].shift(2)

m = smf.ols("deaths_week ~ cal_lag1 + cal_lag2 + C(woy) + C(year)", data=x).fit()
print("cal_lag1 coef:", m.params["cal_lag1"], "p:", m.pvalues["cal_lag1"])
print("cal_lag2 coef:", m.params["cal_lag2"], "p:", m.pvalues["cal_lag2"])

# lead test
x["cal_lead1"] = x["calima_proxy_score_v2"].shift(-1)
x["cal_lead2"] = x["calima_proxy_score_v2"].shift(-2)

mL1 = smf.ols("deaths_week ~ cal_lead1 + C(woy) + C(year)", data=x).fit()
mL2 = smf.ols("deaths_week ~ cal_lead2 + C(woy) + C(year)", data=x).fit()
print("lead1 coef:", mL1.params["cal_lead1"], "p:", mL1.pvalues["cal_lead1"])
print("lead2 coef:", mL2.params["cal_lead2"], "p:", mL2.pvalues["cal_lead2"])

cal_lag1 coef: 0.6506215791890368 p: 0.4972596989732367
cal_lag2 coef: 0.5281561430454427 p: 0.5822811978571851
lead1 coef: -1.465862836568612 p: 0.11543763022412908
lead2 coef: -1.5723358097688283 p: 0.09016973203075676


En vez de usarlo continuo, conviértelo en evento por umbral, por ejemplo:
calima_proxy_level_v2 >= 2 (si ese nivel existe como “fuerte”) o calima_proxy_score_v2 > p95 (episodios extremos del score)

In [24]:
df["calima_proxy_level_v2"].value_counts().sort_index()

calima_proxy_level_v2
intense       37
no_calima     76
possible     279
probable      79
Name: count, dtype: int64

In [25]:
x = df[["week_start","deaths_week","calima_proxy_level_v2"]].copy().sort_values("week_start")

x["woy"]  = x["week_start"].dt.isocalendar().week.astype(int)
x["year"] = x["week_start"].dt.year

x["cal_intense"] = (x["calima_proxy_level_v2"] == "intense").astype(int)

# lags
for L in [0,1,2]:
    x[f"calI_lag{L}"] = x["cal_intense"].shift(L)

# modelo lags juntos (como hicimos con CAP)
m = smf.ols("deaths_week ~ calI_lag1 + calI_lag2 + C(woy) + C(year)", data=x).fit()
print("lag1:", round(m.params["calI_lag1"],2), "p:", m.pvalues["calI_lag1"])
print("lag2:", round(m.params["calI_lag2"],2), "p:", m.pvalues["calI_lag2"])

# lead test
x["calI_lead1"] = x["cal_intense"].shift(-1)
x["calI_lead2"] = x["cal_intense"].shift(-2)

mL1 = smf.ols("deaths_week ~ calI_lead1 + C(woy) + C(year)", data=x).fit()
mL2 = smf.ols("deaths_week ~ calI_lead2 + C(woy) + C(year)", data=x).fit()
print("lead1:", round(mL1.params["calI_lead1"],2), "p:", mL1.pvalues["calI_lead1"])
print("lead2:", round(mL2.params["calI_lead2"],2), "p:", mL2.pvalues["calI_lead2"])

lag1: 2.29 p: 0.41523015202113805
lag2: -0.51 p: 0.8565880897518444
lead1: 1.58 p: 0.562541948036306
lead2: 0.28 p: 0.9194230927172772


In [26]:
x = df.copy()
x = x[x["week_start"] >= "2018-01-01"].dropna(subset=["cap_dust_yellow_plus_week"])

pd.crosstab(x["cap_dust_yellow_plus_week"], x["calima_proxy_level_v2"])

calima_proxy_level_v2,intense,no_calima,possible,probable
cap_dust_yellow_plus_week,,,,
0.0,14,60,193,50
1.0,15,0,6,4


In [27]:
x = df[["week_start","deaths_week","cap_dust_yellow_plus_week","calima_proxy_level_v2"]].copy()
x = x.sort_values("week_start")
x = x[x["week_start"] >= "2018-01-01"].dropna(subset=["cap_dust_yellow_plus_week","calima_proxy_level_v2","deaths_week"])

x["woy"]  = x["week_start"].dt.isocalendar().week.astype(int)
x["year"] = x["week_start"].dt.year
x["cap"]  = (x["cap_dust_yellow_plus_week"] == 1).astype(int)
x["intense"] = (x["calima_proxy_level_v2"] == "intense").astype(int)

# separar "intense con CAP" vs "intense sin CAP"
x["I_cap"]   = ((x["intense"] == 1) & (x["cap"] == 1)).astype(int)
x["I_nocap"] = ((x["intense"] == 1) & (x["cap"] == 0)).astype(int)

# lags
for L in [1,2]:
    x[f"I_cap_lag{L}"]   = x["I_cap"].shift(L)
    x[f"I_nocap_lag{L}"] = x["I_nocap"].shift(L)

m = smf.ols("deaths_week ~ I_cap_lag1 + I_cap_lag2 + I_nocap_lag1 + I_nocap_lag2 + C(woy) + C(year)", data=x).fit()

print("I_cap_lag1:", round(m.params["I_cap_lag1"],2), "p:", m.pvalues["I_cap_lag1"])
print("I_cap_lag2:", round(m.params["I_cap_lag2"],2), "p:", m.pvalues["I_cap_lag2"])
print("I_nocap_lag1:", round(m.params["I_nocap_lag1"],2), "p:", m.pvalues["I_nocap_lag1"])
print("I_nocap_lag2:", round(m.params["I_nocap_lag2"],2), "p:", m.pvalues["I_nocap_lag2"])

I_cap_lag1: 7.07 p: 0.10984142149824964
I_cap_lag2: 7.9 p: 0.07700298724440169
I_nocap_lag1: -1.83 p: 0.7075350304518959
I_nocap_lag2: -6.74 p: 0.14018770658201707


### Qué está pasando

* **“intense & CAP=1”** (`I_cap`) tiene señal **positiva**:

  * lag1 **+7.07** (p=0.11)
  * lag2 **+7.90** (p=0.077) → casi, pero no llega a 0.05
* **“intense & CAP=0”** (`I_nocap`) no tiene señal (y hasta sale negativa):

  * lag1 **-1.83** (p=0.71)
  * lag2 **-6.74** (p=0.14)

### Interpretación

La parte “útil” del proxy calima **viene de cuando coincide con CAP**.
Cuando el proxy dice “intense” pero CAP no lo confirma, **no ves el aumento de muertes** (al menos aquí).

Eso explica por qué “intense” a secas salía nulo: estaba mezclando dos tipos de semanas.

### Por qué CAP sigue siendo mejor

Porque CAP parece estar señalando el subconjunto de eventos que sí tienen el patrón lag1–lag2 fuerte (el que encontraste antes).

### Recomendación para definir episodios, ya con evidencia

Para análisis principal de mortalidad en GC:

1. **Episodio principal:** rachas de `cap_dust_yellow_plus_week==1` (2018+)
2. **Severidad del episodio:** dentro de esos episodios, usa `PM10` y/o `calima_proxy_score_v2` como intensidad
3. **Proxy sin CAP (pre-2018):** si se quiere extender atrás, usar `calima_proxy_level_v2=="intense"` pero **marca incertidumbre** porque no está validado por CAP.


1. **¿Cuál es la diferencia?** (media/mediana)
2. **¿Cuánta incertidumbre tiene?** (IC / bootstrap)
3. **¿Cuándo NO debo confiar?** (n pequeño, colas, episodios, dependencia)

---

## 1) Media vs mediana: qué son y cuándo usar cada una

### Media (mean)

* Es el “promedio clásico”.
* **Se mueve mucho** si hay valores extremos (colas/outliers).

Útil cuando:

* la distribución es más o menos simétrica
* los extremos son raros y no dominan
* te interesa el “peso total” (p.ej. costes, carga total)

### Mediana (median)

* El valor “del medio” (50% por debajo, 50% por encima).
* **Aguanta bien** outliers.

Útil cuando:

* hay colas fuertes (tu PM10 lo es)
* n es pequeño y un episodio puede torcerte el promedio
* quieres “valor típico”

**Regla rápida:**

* Si **mean ≈ median** → suelen contar una historia parecida.
* Si **mean > median** (cola derecha) → la media está “tirada” por picos altos.
* Reporta **las dos** cuando sospechas colas/episodios.

---

## 2) Comparar grupos: qué diferencia reporto

Supón:

* Grupo 0 = no episodio
* Grupo 1 = episodio (CAP yellow+, PM95, etc.)

Tienes varias “diferencias” posibles:

### A) Diferencia de medias

[
\Delta_{\text{mean}} = \bar{y}_1 - \bar{y}_0
]
Interpretación: “en promedio, el grupo 1 tiene X más”.

### B) Diferencia de medianas

[
\Delta_{\text{median}} = \text{median}(y_1) - \text{median}(y_0)
]
Interpretación: “el valor típico cambia X”.

### C) Diferencia robusta alternativa (bonus útil)

* **Trimmed mean** (media recortada 10%): menos sensible a extremos.
* O comparar **p75/p90** si te importa la cola.

---

## 3) Incertidumbre: qué es un IC y por qué importa

Una diferencia (ej. +10 muertes) sin incertidumbre es peligrosa.

Un **intervalo de confianza (IC 95%)** es:

> un rango de valores plausibles para la diferencia, dado el ruido de muestreo.

Si tu IC para Δ es:

* **[+2, +18]** → sugiere efecto positivo consistente.
* **[-5, +12]** → podría ser positivo o negativo: evidencia débil.

### Importante

IC ≠ “probabilidad de que sea verdad”.
Pero te dice si tu estimación es **precisa** o es humo.

---

## 4) Bootstrapping: la idea (sin matemática rara)

Bootstrap = “simular el muestreo” con tus datos:

1. re-muestreas con reemplazo dentro de cada grupo
2. calculas Δ (media o mediana)
3. repites 1000–10000 veces
4. miras la distribución de Δ
5. el IC 95% suele ser percentil 2.5% y 97.5%

Por qué es útil:

* no necesita suponer normalidad
* funciona bien con medianas y distribuciones raras (como PM10)

---

## 5) Cuándo NO confiar (tu checklist anti-autoengaño)

### (1) n pequeño en el grupo “evento”

Si tienes 25 semanas CAP=1, y encima episodios son rachas:

* tu “n efectivo” puede ser menor.

**Síntoma:** IC enorme, resultados que cambian mucho al quitar 1 episodio.

### (2) Datos con colas extremas / outliers

PM10 en GC: max enorme → la media puede ser poco estable.
Preferir:

* mediana / trimmed mean / bootstrap.

### (3) Dependencia temporal (semanas no son independientes)

Si hay rachas, “semanas” no son i.i.d.
Bootstrap simple por filas puede sobreestimar evidencia.

Solución más seria (cuando toque):

* bootstrap por **bloques** (block bootstrap) o análisis por **episodios** (no semanas).

### (4) Multiplicidad

Si pruebas 20 cosas (lags, umbrales, islas) alguna “sale” por azar.
Solución:

* predefinir qué pruebas son principales (p.ej. lag1–lag2 + CAP)

In [28]:
# bootstrap sobre Δ (media y mediana) de deaths_anom entre CAP_lag2=1 vs 0
import numpy as np
import pandas as pd

# --- prep ---
x = df[["week_start","deaths_week","cap_dust_yellow_plus_week"]].copy().sort_values("week_start")
x = x[x["week_start"] >= "2018-01-01"].dropna()

# lag2 de CAP
x["CAP_lag2"] = (x["cap_dust_yellow_plus_week"] == 1).shift(2).astype("float")  # float por NaNs al principio
x = x.dropna(subset=["CAP_lag2"])
x["CAP_lag2"] = x["CAP_lag2"].astype(int)

# anomalía por semana del año (más fino que mes)
woy = x["week_start"].dt.isocalendar().week.astype(int)
x["deaths_anom"] = x["deaths_week"] - x.groupby(woy)["deaths_week"].transform("mean")

g1 = x.loc[x["CAP_lag2"]==1, "deaths_anom"].to_numpy()
g0 = x.loc[x["CAP_lag2"]==0, "deaths_anom"].to_numpy()

print("n0, n1:", len(g0), len(g1))
print("Δ mean (raw):", g1.mean() - g0.mean())
print("Δ median (raw):", np.median(g1) - np.median(g0))

# --- bootstrap ---
rng = np.random.default_rng(42)
B = 5000

boot_mean = np.empty(B)
boot_median = np.empty(B)

for b in range(B):
    s1 = rng.choice(g1, size=len(g1), replace=True)
    s0 = rng.choice(g0, size=len(g0), replace=True)
    boot_mean[b] = s1.mean() - s0.mean()
    boot_median[b] = np.median(s1) - np.median(s0)

ci_mean = np.quantile(boot_mean, [0.025, 0.975])
ci_med  = np.quantile(boot_median, [0.025, 0.975])

print("IC95 Δmean:", ci_mean)
print("IC95 Δmedian:", ci_med)

n0, n1: 315 25
Δ mean (raw): 17.110506424792135
Δ median (raw): 14.166666666666657
IC95 Δmean: [ 9.22433976 25.49037113]
IC95 Δmedian: [ 8.5702381  26.03214286]


**Interpretación:** en semanas donde CAP fue 1 hace 2 semanas, las muertes (desestacionalizadas por woy) están claramente por encima del baseline. Y no depende solo de outliers, porque mediana también es positiva y con IC > 0.

### ¿El efecto Δ (lag2) sigue existiendo si quitamos el episodio CAP más largo?

In [29]:

# --- prep base ---
x = df[["week_start","deaths_week","cap_dust_yellow_plus_week"]].copy().sort_values("week_start")
x = x[x["week_start"] >= "2018-01-01"].dropna()

# CAP flag
x["cap"] = (x["cap_dust_yellow_plus_week"] == 1).astype(int)

# identificar episodios CAP (rachas cap==1)
cap_start = (x["cap"] == 1) & (x["cap"].shift(1, fill_value=0) == 0)
x["cap_ep_id"] = cap_start.cumsum()
x.loc[x["cap"] == 0, "cap_ep_id"] = pd.NA

# tabla episodios + encontrar el más largo
cap_eps = (
    x.dropna(subset=["cap_ep_id"])
     .groupby("cap_ep_id")
     .agg(weeks=("week_start","size"),
          start=("week_start","min"),
          end=("week_start","max"))
     .sort_values("weeks", ascending=False)
)

longest_id = cap_eps.index[0]
print("Longest CAP episode id:", int(longest_id))
print(cap_eps.head(5))

# --- función para calcular Δ lag2 sobre deaths_anom (woy) ---
def delta_lag2(data):
    d = data.copy()
    d["CAP_lag2"] = d["cap"].shift(2)
    d = d.dropna(subset=["CAP_lag2"])
    d["CAP_lag2"] = d["CAP_lag2"].astype(int)

    woy = d["week_start"].dt.isocalendar().week.astype(int)
    d["deaths_anom"] = d["deaths_week"] - d.groupby(woy)["deaths_week"].transform("mean")

    g1 = d.loc[d["CAP_lag2"]==1, "deaths_anom"].to_numpy()
    g0 = d.loc[d["CAP_lag2"]==0, "deaths_anom"].to_numpy()

    return {
        "n0": len(g0),
        "n1": len(g1),
        "d_mean": float(g1.mean() - g0.mean()),
        "d_median": float(np.median(g1) - np.median(g0)),
    }

# 1) Δ con todos los datos
all_res = delta_lag2(x)
print("\nALL:", all_res)

# 2) Δ quitando el episodio CAP más largo (todas sus semanas)
x_drop = x[x["cap_ep_id"] != longest_id].copy()
drop_res = delta_lag2(x_drop)
print("\nDROP longest episode:", drop_res)

# 3) cuánto cambia
print("\nChange in Δmean:", drop_res["d_mean"] - all_res["d_mean"])
print("Change in Δmedian:", drop_res["d_median"] - all_res["d_median"])

Longest CAP episode id: 9
           weeks      start        end
cap_ep_id                             
9.0            4 2022-01-03 2022-01-24
4.0            2 2020-02-17 2020-02-24
2.0            2 2018-12-17 2018-12-24
16.0           2 2023-09-25 2023-10-02
1.0            1 2018-09-10 2018-09-10

ALL: {'n0': 315, 'n1': 25, 'd_mean': 17.110506424792135, 'd_median': 14.166666666666657}

DROP longest episode: {'n0': 315, 'n1': 21, 'd_mean': 14.214240362811786, 'd_median': 13.571428571428584}

Change in Δmean: -2.8962660619803486
Change in Δmedian: -0.5952380952380736


### Re-muestreamos episodios completos (rachas CAP) como unidades.

In [30]:
# --- prep base ---
x = df[["week_start","deaths_week","cap_dust_yellow_plus_week"]].copy().sort_values("week_start")
x = x[x["week_start"] >= "2018-01-01"].dropna()

x["cap"] = (x["cap_dust_yellow_plus_week"] == 1).astype(int)

# anomalía por semana del año (en el dataset base)
woy = x["week_start"].dt.isocalendar().week.astype(int)
x["deaths_anom"] = x["deaths_week"] - x.groupby(woy)["deaths_week"].transform("mean")

# identificar episodios CAP (rachas cap==1)
cap_start = (x["cap"] == 1) & (x["cap"].shift(1, fill_value=0) == 0)
x["cap_ep_id"] = cap_start.cumsum()
x.loc[x["cap"] == 0, "cap_ep_id"] = pd.NA

episodes = (
    x.dropna(subset=["cap_ep_id"])
     .groupby("cap_ep_id")
     .agg(start=("week_start","min"),
          end=("week_start","max"),
          weeks=("week_start","size"))
     .reset_index()
)

print("CAP episodes:", len(episodes))
print("Episode weeks distribution:\n", episodes["weeks"].value_counts().sort_index())

# --- helper: Δ lag2 sobre un subconjunto de semanas ---
def delta_lag2_from_subset(d):
    d = d.sort_values("week_start").copy()
    d["CAP_lag2"] = d["cap"].shift(2)
    d = d.dropna(subset=["CAP_lag2"])
    d["CAP_lag2"] = d["CAP_lag2"].astype(int)

    g1 = d.loc[d["CAP_lag2"]==1, "deaths_anom"].to_numpy()
    g0 = d.loc[d["CAP_lag2"]==0, "deaths_anom"].to_numpy()

    if len(g1) < 5 or len(g0) < 20:   # guardrail mínimo
        return np.nan, np.nan

    d_mean = float(g1.mean() - g0.mean())
    d_med  = float(np.median(g1) - np.median(g0))
    return d_mean, d_med

# --- bootstrap by episodes ---
rng = np.random.default_rng(42)
B = 3000

boot_mean = np.empty(B)
boot_med  = np.empty(B)

ep_ids = episodes["cap_ep_id"].to_numpy()
n_eps  = len(ep_ids)

# precomputar baseline (semanas cap==0) para que siempre haya "control"
base0 = x[x["cap"] == 0].copy()

for b in range(B):
    # sample episodes with replacement
    sample_ids = rng.choice(ep_ids, size=n_eps, replace=True)

    # concatenar semanas de esos episodios
    sample1 = pd.concat([x[x["cap_ep_id"] == sid] for sid in sample_ids], ignore_index=True)

    # construimos dataset bootstrap: siempre incluye controles (cap=0) + episodios muestreados
    d = pd.concat([base0, sample1], ignore_index=True)

    dm, dmd = delta_lag2_from_subset(d)
    boot_mean[b] = dm
    boot_med[b]  = dmd

# limpiar NaNs (por si alguna iteración quedó sin suficientes semanas g1)
boot_mean = boot_mean[~np.isnan(boot_mean)]
boot_med  = boot_med[~np.isnan(boot_med)]

ci_mean = np.quantile(boot_mean, [0.025, 0.975])
ci_med  = np.quantile(boot_med,  [0.025, 0.975])

print("\nBootstrap-by-episode results")
print("Δmean point (episode-boot mean):", float(np.mean(boot_mean)))
print("IC95 Δmean:", ci_mean)
print("Δmedian point (episode-boot median):", float(np.mean(boot_med)))
print("IC95 Δmedian:", ci_med)
print("bootstrap draws used:", len(boot_mean))

CAP episodes: 19
Episode weeks distribution:
 weeks
1    15
2     3
4     1
Name: count, dtype: int64

Bootstrap-by-episode results
Δmean point (episode-boot mean): 13.281497443802994
IC95 Δmean: [ 3.32144548 21.38165684]
Δmedian point (episode-boot median): 13.237257936507934
IC95 Δmedian: [ 6.85714286 18.95238095]
bootstrap draws used: 3000


### Interpretación limpia

Incluso tratando los eventos como bloques (más conservador), sigue habiendo evidencia de **exceso de muertes (desestacionalizadas, lag2)** asociado a episodios CAP. La magnitud plausible está, grosso modo, entre:

* **~+3 y +21** (media)
* **~+7 y +19** (mediana)

### Cuándo “no confiar” a partir de aquí

Ahora ya no es el típico “n pequeño = basura”. El punto débil principal sería:

* que tu definición de episodio o el lag esté mal alineado (pero lead test salió limpio),
* o que falten confusores importantes (virus/otros contaminantes/meteo más completa).


### Result (Gran Canaria, 2018+): CAP dust episodes and lagged mortality

After controlling for the annual pattern (seasonality) by comparing similar weeks of the year, official dust-alert episodes (**CAP yellow+**) in Gran Canaria are associated with higher weekly deaths **1–2 weeks later**, with the strongest signal at **lag 2**.

This pattern remains under uncertainty checks (episode-level bootstrap) and does not appear in a temporal placebo (**lead test**, using “future CAP” to predict current deaths), which supports that the result is not a simple calendar artifact.